[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/validation/labse-multilingual-smoketest/experiments/labse_english_hebrew_only.ipynb)


# Language-agnostic BERT sentence embeddings — English to Hebrew

LaBSE encodes text into high-dimensional vectors. It is trained to align translated sentences across languages, including English and Hebrew.

**Only task:** Compare the **6 English-only employer labels** against the **76 Hebrew-containing employer labels** observed in the survey. Suggest an existing Hebrew label only when its cosine similarity is above a conservative threshold. Do **not** compare English with English, fine-tune, cluster, create graphs, or modify Hebrew labels.

**Safety:** Three English/Hebrew aliases are already verified by their names. Other proposed matches remain unchanged until verified. We test the name-only input first; we use equivalent workplace sentences *only if the measured matching accuracy on the verified aliases improves without adding errors*. A missed match is preferable to a mistaken replacement.

**Method limits:** The three verified pairs are a diagnostic sample, not sufficient to validate a universal threshold. LaBSE calculates text similarity, not verified corporate identity. A model download is required on the first run. One-click: **Runtime → Run all** in free Colab (CPU).


In [ ]:
# 1. Load the public PyTorch model. No token or paid API is needed.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--quiet", "sentence-transformers==5.1.2"
    ])

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from IPython.display import display


# 2. Use the actual employer labels in the 200-row survey.
# Match English with Hebrew only; retain original values.
english = ["COGNYTE","EMILIO","INTEL","KPMG","MICROSOFT","NVIDIA"]
hebrew = ["4 עונות","א.א. סיעוד","אדרים","אוניברסיטת באר שבע","אוניברסיטת תל אביב","אוסם","אחים כהן","איחים חג'ג'","אינטל","אמדוקס","אמיר כהן","בזק","בית חולים הדסה עין כרם","בית מלון","בית ספר תיכון","בנק אוצר החייל","בנק דיסקונט","בנק הפועלים","בנק לאומי","בנק מזרחי","גידול תפוזים ","גידול תפוחים ","גן חדרה ","דואר ישראל","הרודס","חברת הייטק","חח\"י","חנות בגדים של מעצבים","חנות פולגת","יועצץ מחשוב","יועצת IT עצמאית","ישרוטל","למ\"ס","מוסך א.א.","מוסך אחים.","מיקרוסופט","מכללת חדרה","מלון אקאדיה","מלון דן","מסעדה","מסעדת טעמים","מסעדת נונה","מסעדת עזורה","מקורות","משטרה","משרד החינוך","משרד הפנים","משרד ממשלתי","נבידיה","סולל בונה","סומך חייקין","סופר פארם","סופרפארם","עובד במפעל  ","עובד במפעל   הבקבוקים","עובד במפעל של הקיבוץ","עירייה","עיריית חדרה","עיריית ירושלים","עיריית תל אביב","עיריית תל אילת","עלית","פרשקובסקי","קוסמטיקאית ","קוסמטיקאית עצמאית","קניון עזריאלי","קריית הממשלה","רמי לוי","רשת בי פארם","שופר סל","שופרסל","תיאטרון ","תיאטרון הבימה","תיאטרון חיפה","תיאטרון ירושלים","תל אביב יפו - מסעדה"]

# Three known English/Hebrew aliases in the source data.
# Other English labels have no VERIFIED Hebrew equivalent here.
verified = {"INTEL":"אינטל","MICROSOFT":"מיקרוסופט","NVIDIA":"נבידיה"}

assert len(english) == 6 and len(hebrew) == 76
assert all(value in hebrew for value in verified.values())


# 3. Compare plain labels with labels placed inside matching sentences.
# Sentence context is a hypothesis to test, not an automatic improvement.
model = SentenceTransformer("sentence-transformers/LaBSE", device="cpu")

def score_matrix(left, right):
    vec = model.encode(
        left + right,
        batch_size=8,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    return vec[:len(left)] @ vec[len(left):].T

plain = score_matrix(english, hebrew)
sentences = score_matrix(
    [f"I work at {name}." for name in english],
    [f"אני עובד ב{name}." for name in hebrew],
)


# 4. Choose the best Hebrew candidate only when its score and lead
# over the second-best candidate are both sufficient.
# This reduces uncertain replacements, at the cost of missed matches.
MARGIN = 0.08
THRESHOLDS = np.round(np.arange(0.50, 1.00, 0.05), 2)

def matches(scores, threshold):
    result = {}

    for i, name in enumerate(english):
        order = np.argsort(scores[i])[::-1]
        best, second = int(order[0]), int(order[1])

        is_clear = (
            scores[i, best] > threshold
            and scores[i, best] - scores[i, second] >= MARGIN
        )
        result[name] = hebrew[best] if is_clear else None

    return result


# 5. Select the lowest threshold with no incorrect replacements
# among the three verified aliases; then maximize correct matches.
# This is a small diagnostic check, not population-wide validation.
def assess(scores):
    trials = []

    for threshold in THRESHOLDS:
        proposed = matches(scores, threshold)
        correct = sum(proposed[name] == target for name, target in verified.items())
        wrong = sum(
            proposed[name] is not None and proposed[name] != target
            for name, target in verified.items()
        )
        trials.append((wrong, -correct, float(threshold), proposed))

    return min(trials, key=lambda item: item[:3])

plain_test = assess(plain)
sentence_test = assess(sentences)

# Use sentence context ONLY if it improves correct aliases without
# increasing incorrect replacements. Otherwise retain the plain model.
use_sentences = (
    sentence_test[0] <= plain_test[0]
    and -sentence_test[1] > -plain_test[1]
)
selected = sentence_test if use_sentences else plain_test
scores = sentences if use_sentences else plain

threshold = selected[2]
proposed = selected[3]

print("Plain names:   wrong =", plain_test[0],
      "| correct =", -plain_test[1], "| threshold =", plain_test[2])
print("With context:  wrong =", sentence_test[0],
      "| correct =", -sentence_test[1], "| threshold =", sentence_test[2])
print("Selected:", "SENTENCES" if use_sentences else "PLAIN NAMES")
print("Threshold:", threshold, "| Margin:", MARGIN)


# 6. Show all six English labels, including rejected proposals.
# An unverified proposal is never applied, even with a high score.
rows = []
replacements = {}

for i, name in enumerate(english):
    order = np.argsort(scores[i])[::-1]
    first, second = int(order[0]), int(order[1])
    suggestion = proposed[name]

    verified_match = suggestion is not None and verified.get(name) == suggestion
    if verified_match:
        replacements[name] = suggestion

    rows.append({
        "English": name,
        "Nearest Hebrew": hebrew[first],
        "Cosine": round(float(scores[i, first]), 4),
        "Margin": round(float(scores[i, first] - scores[i, second]), 4),
        "Decision": (
            "REPLACE (verified)" if verified_match
            else "KEEP ENGLISH (unverified or below threshold)"
        ),
    })

display(pd.DataFrame(rows))

# 7. Preserve every original Hebrew name and any unmatched English name.
# Save the resulting label mapping without modifying the survey workbook.
result = pd.DataFrame({
    "Original_ShemAvoda": english + hebrew
})
result["Normalized_ShemAvoda"] = (
    result["Original_ShemAvoda"].replace(replacements)
)
result.to_csv("labse_english_to_hebrew_labels.csv",
              index=False, encoding="utf-8-sig")

print("Verified replacements:", replacements)
print("English labels kept unchanged:", len(english) - len(replacements))
print("HEBREW_ENGLISH_MATCHING_COMPLETE")


## Department fields — full English-to-Hebrew similarity test

**Source:** `DUMMY_DATASET.xlsm` (200 survey records, checked 2026-11-03). The test uses **every distinct, nonblank label** observed in `ShemMachlaka` (61 values) and `SugMachlaka` (35 values). Together there are 83 distinct labels, including **79 Hebrew-containing** values and four English-only values: `R&D`, `IT`, `HR`, `DELIVERY`.

**Test:** Compare each English term against **all 79 Hebrew candidates** from both columns using the LaBSE model already loaded above. Display `R&D` first, its complete 79-candidate ranking, every candidate scoring strictly **greater than 0.50**, and a ranking for the other English labels. Export the entire **316-score** result table as a CSV.

**Important:** An above-0.50 score is a candidate for review, **not automatically a correct translation**. No original value is modified, and this test does not fit TF-IDF or fine-tune the model. The source labels are embedded as an auditable snapshot so free Google Colab requires no Drive authentication.


In [ ]:
# Reuse the pretrained LaBSE model from the first code cell.
# These are ALL distinct nonempty values in the two 200-row survey columns.
department_values = {
    "ShemMachlaka": ["DELIVERY","HR","IT","R&D","אריזה","אשראי","ביולוגיה","הנדסה","הנדסת תוכנה","הנהלת חשבונות","הצגות","השקעות","התקנת מזגנים","התקנת מטבחים","חבלן","חדר נקי","חינוך","חסוי","חפירות","חשמלאות","יומנאי","ייצור צ'יפים","כ\"א","כימיה","מאקרו כלכלה","מדע נתונים","מורה","מחלקת רווחה","מחלקת תרבות","מחסן","מחקר ופיתוח","מחשוב","מטבח","מיקרו כלכלה","מכונאות","מכירות","מנהל","מסעדה","משק","סיעוד","סקרים","עבודות חשמל","עו\"ש","עובד משק","עובד ניקיון","עסקים","פלילית","פס ייצור","פרטיים","קוסמטיקה וטיפוח","קופה","רדיואתרפיה","רואה חשבון","רוקח","רכש","שיווק","שמירה","תיקון תקלות","תכנה","תמיכה בלקוחות","תמיכה בתקלות"],
    "SugMachlaka": ["IT","R&D","איסוף נתונים בשטח","אריזה","אשראי","בדיקת דירוג אשראי","גננת","הזמנת ציוד","הנדסה","חישובי שכר לעובדים","חפירות","חשבת שכר לעובדים","טיפול באנשים עם בעיות","טיפול בבקשת הלואות","טיפול בחולי סרטן","טיפול בלקוחות עסקיים","טיפול בנתוני מאקרו","טיפול בנתוני מיקרו","יעוץ השקעות ללקוחות","כנ\"ל","מדע נתונים","מחלקת כ\"א","מחסן","מחקר ופיתוח","משאבי אנוש","משחק","מתמתיקה","סיעוד","קבלת קהל","קוסמטיקה","רואה חשבון","רוקח","שיווק","שיווק עוגות","תמיכה במערכות מחשוב של הבנק"],
}

# Collect English labels and Hebrew candidate labels across BOTH columns.
# A label appearing in both columns is encoded only once.
all_values = sorted(set(sum(department_values.values(), [])))
has_hebrew = lambda s: any("\u0590" <= char <= "\u05ff" for char in s)

english_terms = ["R&D"] + sorted(
    value for value in all_values
    if not has_hebrew(value) and value != "R&D"
)
hebrew_terms = [value for value in all_values if has_hebrew(value)]

assert len(english_terms) == 4
assert len(hebrew_terms) == 79
assert len(all_values) == 83
assert "מחקר ופיתוח" in hebrew_terms

print("Source: 200 survey records, 2 department columns.")
print("English labels:", english_terms)
print("Unique Hebrew candidates:", len(hebrew_terms))

# Measure every English–Hebrew cosine similarity using the same LaBSE model.
# No fine-tuning, translation API, or model reload is needed.
embeddings = model.encode(
    english_terms + hebrew_terms,
    batch_size=8,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
)
similarities = embeddings[:len(english_terms)] @ embeddings[len(english_terms):].T

# Keep the source column(s) for each Hebrew candidate, so the ranking
# shows whether a label came from ShemMachlaka, SugMachlaka, or both.
records = []
for i, english_name in enumerate(english_terms):
    for j, hebrew_name in enumerate(hebrew_terms):
        records.append({
            "English": english_name,
            "Hebrew": hebrew_name,
            "Cosine": float(similarities[i, j]),
            "ShemMachlaka": hebrew_name in department_values["ShemMachlaka"],
            "SugMachlaka": hebrew_name in department_values["SugMachlaka"],
            "Above_0.50": bool(similarities[i, j] > 0.50),
        })

results = pd.DataFrame(records)
assert len(results) == 4 * 79

# First: R&D against ALL 79 Hebrew values, highest cosine first.
# Show all >0.50 matches separately; do not replace any label.
for term in english_terms:
    ranked = (
        results[results["English"] == term]
        .sort_values("Cosine", ascending=False)
        .reset_index(drop=True)
    )
    ranked.index = ranked.index + 1
    above = ranked[ranked["Above_0.50"]]

    print(f"\n{term}: top Hebrew candidate = {ranked.iloc[0]['Hebrew']}"
          f" (cosine={ranked.iloc[0]['Cosine']:.4f})")
    print(f"Matches strictly above 0.50: {len(above)} / {len(ranked)}")
    print("ALL candidates >0.50:")
    display(above)

    if term == "R&D":
        print("R&D — COMPLETE ranking of all 79 Hebrew values:")
        display(ranked)
        reference = ranked[ranked["Hebrew"] == "מחקר ופיתוח"]
        print("Verified translation reference: R&D ↔ מחקר ופיתוח")
        display(reference)
    else:
        print("Top 10 candidates:")
        display(ranked.head(10))

# Save every measured candidate (including those below the threshold).
results.sort_values(
    ["English", "Cosine"], ascending=[True, False]
).to_csv(
    "labse_department_english_hebrew_all_316_scores.csv",
    index=False,
    encoding="utf-8-sig",
)

print("DEPARTMENT_COMPARISON_SUCCESS — 316 scores; no values changed.")
